In [1]:
import pandas as pd
import numpy as np
from datetime import datetime


df = pd.read_csv('employee_data_with_feedback.csv')

df['StartDate'] = pd.to_datetime(df['StartDate'], dayfirst=True, errors='coerce')
df['DOB'] = pd.to_datetime(df['DOB'], dayfirst=True, errors='coerce')

current_date = pd.to_datetime('2025-01-01') 
df['Tenure_Years'] = ((current_date - df['StartDate']).dt.days / 365).round(1)

df['Age'] = ((current_date - df['DOB']).dt.days / 365).round(1)

df['Attrition_Target'] = df['ExitDate'].apply(lambda x: 0 if pd.isna(x) else 1)

df = df.drop(columns=['FirstName', 'LastName', 'ADEmail', 'DOB', 'ExitDate', 'TerminationDescription', 'LocationCode', 'EmployeeStatus'])

print("Row before data pre processing :")
print(df.head().to_markdown(index=False, numalign="left", stralign="left"))

Row before data pre processing :
| EmpID   | StartDate           | Title                   | Supervisor      | BusinessUnit   | EmployeeType   | PayZone   | EmployeeClassificationType   | TerminationType   | DepartmentType   | Division             | State   | JobFunctionDescription   | GenderCode   | RaceDesc   | MaritalDesc   | Performance Score   | Current Employee Rating   | Hygiene   | Safety Measures   | Food Quality   | Accommodation   | Overall Satisfaction   | Tenure_Years   | Age   | Attrition_Target   |
|:--------|:--------------------|:------------------------|:----------------|:---------------|:---------------|:----------|:-----------------------------|:------------------|:-----------------|:---------------------|:--------|:-------------------------|:-------------|:-----------|:--------------|:--------------------|:--------------------------|:----------|:------------------|:---------------|:----------------|:-----------------------|:---------------|:------|:----------------

C:\Users\anshu\AppData\Local\Temp\ipykernel_22620\2930098177.py:8: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df['StartDate'] = pd.to_datetime(df['StartDate'], dayfirst=True, errors='coerce')


In [2]:
# संतुष्टि/गुणवत्ता के स्तरों को मैप करना
satisfaction_map = {
    'very low': 1, 'Low': 2, 'Medium': 3, 'High': 4,
    'Poor': 1, 'Average': 2, 'Good': 3, 'Excellent': 4
}
performance_map = {
    'Exceeds': 4, 'Fully Meets': 3, 'Needs Improvement': 2, 'NA': 1, 'PIP': 0
}

df['Overall_Satisfaction_Encoded'] = df['Overall Satisfaction'].map(satisfaction_map)
df['Hygiene_Encoded'] = df['Hygiene'].map(satisfaction_map)
df['Food_Quality_Encoded'] = df['Food Quality'].map(satisfaction_map)
df['Accommodation_Encoded'] = df['Accommodation'].map(satisfaction_map)
df['Performance_Score_Encoded'] = df['Performance Score'].map(performance_map)


df = df.drop(columns=['Overall Satisfaction', 'Hygiene', 'Food Quality', 'Accommodation', 'Performance Score'])

In [3]:

categorical_cols = df.select_dtypes(include=['object']).columns


df_encoded = pd.get_dummies(df, columns=categorical_cols, drop_first=True)

print("\nEncoding के बाद डेटासेट के कॉलम की संख्या:", df_encoded.shape[1])


Encoding के बाद डेटासेट के कॉलम की संख्या: 3160


In [4]:

X_attrition = df_encoded.drop(columns=['Attrition_Target', 'Current Employee Rating', 'Overall_Satisfaction_Encoded', 'Performance_Score_Encoded'])
Y_attrition = df_encoded['Attrition_Target']

print("Attrition Features (X_attrition) Shape:", X_attrition.shape)

Attrition Features (X_attrition) Shape: (3000, 3156)


In [5]:

X_satisfaction = df_encoded.drop(columns=['Overall_Satisfaction_Encoded', 'Attrition_Target', 'Current Employee Rating', 'Performance_Score_Encoded'])
Y_satisfaction = df_encoded['Overall_Satisfaction_Encoded']

print("Satisfaction Features (X_satisfaction) Shape:", X_satisfaction.shape)

Satisfaction Features (X_satisfaction) Shape: (3000, 3156)


In [6]:

X_performance = df_encoded.drop(columns=['Performance_Score_Encoded', 'Attrition_Target', 'Current Employee Rating', 'Overall_Satisfaction_Encoded'])
Y_performance = df_encoded['Performance_Score_Encoded']

print("Performance Features (X_performance) Shape:", X_performance.shape)

Performance Features (X_performance) Shape: (3000, 3156)


In [7]:

Y_attrition = df_encoded['Attrition_Target']
X_attrition = df_encoded.drop(columns=['Attrition_Target']) 


Y_rating = df_encoded['Current Employee Rating']
X_rating = df_encoded.drop(columns=['Current Employee Rating']) 
Y_satisfaction = df_encoded['Overall_Satisfaction_Encoded']
X_satisfaction = df_encoded.drop(columns=['Overall_Satisfaction_Encoded']) 
Y_performance = df_encoded['Performance_Score_Encoded']
X_performance = df_encoded.drop(columns=['Performance_Score_Encoded']) 
print("All X and Y are succesfully executed")

All X and Y are succesfully executed


In [8]:
from sklearn.model_selection import train_test_split


X_train_att, X_test_att, Y_train_att, Y_test_att = train_test_split(
    X_attrition, Y_attrition, test_size=0.2, random_state=42
)

X_train_rate, X_test_rate, Y_train_rate, Y_test_rate = train_test_split(
    X_rating, Y_rating, test_size=0.2, random_state=42
)

X_train_sat, X_test_sat, Y_train_sat, Y_test_sat = train_test_split(
    X_satisfaction, Y_satisfaction, test_size=0.2, random_state=42
)

X_train_perf, X_test_perf, Y_train_perf, Y_test_perf = train_test_split(
    X_performance, Y_performance, test_size=0.2, random_state=42
)

print("\nData Split for all the models is successful")


Data Split for all the models is successful


In [9]:
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, mean_squared_error, r2_score
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
import numpy as np
import pandas as pd

NUMERICAL_FEATURES = ['Tenure_Years', 'Age']

def scale_data(X_train, X_test):
    """ट्रेन और टेस्ट डेटा पर केवल संख्यात्मक फीचर्स को स्केल करता है।"""
    scaler = StandardScaler()
    
    num_cols = [col for col in NUMERICAL_FEATURES if col in X_train.columns]
    
    
    X_train[num_cols] = scaler.fit_transform(X_train[num_cols])
    
    X_test[num_cols] = scaler.transform(X_test[num_cols])
    
    return X_train, X_test

X_train_att, X_test_att = scale_data(X_train_att, X_test_att)
X_train_rate, X_test_rate = scale_data(X_train_rate, X_test_rate)
X_train_sat, X_test_sat = scale_data(X_train_sat, X_test_sat)
X_train_perf, X_test_perf = scale_data(X_train_perf, X_test_perf)

print("All numeric features are succesfully scaled")

All numeric features are succesfully scaled


In [10]:

def remove_datetime_cols(df):
    """DataFrame से DateTime DType वाले सभी कॉलम हटाता है।"""
    datetime_cols = df.select_dtypes(include=['datetime64']).columns
    return df.drop(columns=datetime_cols, errors='ignore')

X_train_att_clean = remove_datetime_cols(X_train_att_clean)
X_test_att_clean = remove_datetime_cols(X_test_att_clean)

X_train_rate = remove_datetime_cols(X_train_rate)
X_test_rate = remove_datetime_cols(X_test_rate)

X_train_sat = remove_datetime_cols(X_train_sat)
X_test_sat = remove_datetime_cols(X_test_sat)

X_train_perf = remove_datetime_cols(X_train_perf)
X_test_perf = remove_datetime_cols(X_test_perf)


print("डेटटाइम फीचर्स हटाने के बाद X_train_att_clean का नया आकार:", X_train_att_clean.shape)

rf_att = RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42, class_weight='balanced')
rf_att.fit(X_train_att_clean, Y_train_att)

Y_pred_att = rf_att.predict(X_test_att_clean)

accuracy_att = accuracy_score(Y_test_att, Y_pred_att)
f1_att = f1_score(Y_test_att, Y_pred_att, zero_division=0)

print("\n--- Model 1: Employee attrition prediction  (Random Forest) ---")
print(f"**(Accuracy)** (After Fix tthe data )): {accuracy_att:.4f}")
print(f"**F1 Score** (After fix): {f1_att:.4f}")

NameError: name 'X_train_att_clean' is not defined

In [ ]:
from sklearn.model_selection import GridSearchCV

param_grid_rate = {
    'n_estimators': [50, 100],  # कम एस्टिमेटर्स से शुरू करें
    'max_depth': [5, 8],        # कम गहराई पर ध्यान दें
    'min_samples_split': [2, 5]
}

grid_search_rate = GridSearchCV(
    estimator=RandomForestRegressor(random_state=42),
    param_grid=param_grid_rate,
    cv=3,                      # 3-फोल्ड क्रॉस-वैलिडेशन
    scoring='r2',
    verbose=1
)



In [ ]:
feature_importances_att = pd.Series(rf_att.feature_importances_, index=X_train_att_clean.columns)
top_10_features_att = feature_importances_att.nlargest(10)

print("\n--- Model 1: All 10 important features for attrition ---")
print(top_10_features_att.to_markdown(numalign="left", stralign="left"))


--- मॉडल 1: शीर्ष 10 सबसे महत्वपूर्ण फीचर्स (Attrition के लिए) ---
|                              | 0         |
|:-----------------------------|:----------|
| EmpID                        | 0.0514043 |
| Tenure_Years                 | 0.0267026 |
| Accommodation_Encoded        | 0.0255141 |
| Age                          | 0.0246232 |
| Overall_Satisfaction_Encoded | 0.0195377 |
| Food_Quality_Encoded         | 0.0162816 |
| Title_BI Developer           | 0.0151383 |
| Performance_Score_Encoded    | 0.0130971 |
| Division_Splicing            | 0.0129232 |
| Current Employee Rating      | 0.0121894 |


In [ ]:
import joblib
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
import pandas as pd
import numpy as np


NUMERICAL_FEATURES = ['Tenure_Years', 'Age']

Y_att = df_encoded['Attrition_Target']
X_att = df_encoded.drop(columns=['Attrition_Target'])
Y_rate = df_encoded['Current Employee Rating']
X_rate = df_encoded.drop(columns=['Current Employee Rating'])

Y_sat = df_encoded['Overall_Satisfaction_Encoded']
X_sat = df_encoded.drop(columns=['Overall_Satisfaction_Encoded'])

Y_perf = df_encoded['Performance_Score_Encoded']
X_perf = df_encoded.drop(columns=['Performance_Score_Encoded'])



def remove_datetime_cols(df):
    """DataFrame से DateTime DType वाले सभी कॉलम हटाता है।"""
    df_copy = df.copy() 
    datetime_cols = df_copy.select_dtypes(include=['datetime64']).columns
    return df_copy.drop(columns=datetime_cols, errors='ignore')

def scale_data(X_train, X_test, scaler):
    """ट्रेन और टेस्ट डेटा पर केवल संख्यात्मक फीचर्स को स्केल करता है।"""
    num_cols = [col for col in NUMERICAL_FEATURES if col in X_train.columns]
    if not hasattr(scaler, 'scale_'):
        scaler.fit(X_train[num_cols])
    
    X_train[num_cols] = scaler.transform(X_train[num_cols])
    X_test[num_cols] = scaler.transform(X_test[num_cols])
    return X_train, X_test

# Split the data
X_train_att, X_test_att, Y_train_att, Y_test_att = train_test_split(X_att, Y_att, test_size=0.2, random_state=42)
X_train_rate, X_test_rate, Y_train_rate, Y_test_rate = train_test_split(X_rate, Y_rate, test_size=0.2, random_state=42)
X_train_sat, X_test_sat, Y_train_sat, Y_test_sat = train_test_split(X_sat, Y_sat, test_size=0.2, random_state=42)
X_train_perf, X_test_perf, Y_train_perf, Y_test_perf = train_test_split(X_perf, Y_perf, test_size=0.2, random_state=42)

print("Data Split for all the models is successful")



scaler = StandardScaler()

X_train_att_clean = remove_datetime_cols(X_train_att)
X_test_att_clean = remove_datetime_cols(X_test_att)
termination_cols = [col for col in X_train_att_clean.columns if col.startswith('TerminationType')]
X_train_att_clean = X_train_att_clean.drop(columns=termination_cols, errors='ignore')
X_test_att_clean = X_test_att_clean.drop(columns=termination_cols, errors='ignore')
X_train_att_clean, X_test_att_clean = scale_data(X_train_att_clean, X_test_att_clean, scaler) 
joblib.dump(scaler, 'scaler.joblib')
print("StandardScaler saved.")

X_train_perf_clean = remove_datetime_cols(X_train_perf)
X_test_perf_clean = remove_datetime_cols(X_test_perf)
X_train_perf_clean, X_test_perf_clean = scale_data(X_train_perf_clean, X_test_perf_clean, scaler)

X_train_sat_clean = remove_datetime_cols(X_train_sat)
X_test_sat_clean = remove_datetime_cols(X_test_sat)
X_train_sat_clean, X_test_sat_clean = scale_data(X_train_sat_clean, X_test_sat_clean, scaler)

X_train_rate_clean = remove_datetime_cols(X_train_rate)
X_test_rate_clean = remove_datetime_cols(X_test_rate)
X_train_rate_clean, X_test_rate_clean = scale_data(X_train_rate_clean, X_test_rate_clean, scaler)


rf_att = RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42, class_weight='balanced')
rf_att.fit(X_train_att_clean, Y_train_att)
joblib.dump(rf_att, 'rf_attrition_model.joblib')

rf_perf = RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42)
rf_perf.fit(X_train_perf_clean, Y_train_perf)
joblib.dump(rf_perf, 'rf_performance_model.joblib')
rf_sat = RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42)
rf_sat.fit(X_train_sat_clean, Y_train_sat)
joblib.dump(rf_sat, 'rf_satisfaction_model.joblib')
rf_rate = RandomForestRegressor(n_estimators=100, max_depth=10, random_state=42)
rf_rate.fit(X_train_rate_clean, Y_train_rate)
joblib.dump(rf_rate, 'rf_rating_model.joblib')

print("All 4 models have been successfully trained and saved.")

joblib.dump(X_train_att_clean.columns.tolist(), 'attrition_features.joblib')
joblib.dump(X_train_perf_clean.columns.tolist(), 'performance_features.joblib')
joblib.dump(X_train_sat_clean.columns.tolist(), 'satisfaction_features.joblib')
joblib.dump(X_train_rate_clean.columns.tolist(), 'rating_features.joblib')
print("All features are saved")
print("\n for streamlit development run the app.py file ")

डेटा विभाजित हो गया है।
StandardScaler saved.
सभी 4 मॉडल सफलतापूर्वक प्रशिक्षित और सेव हो गए हैं।
सभी 4 फीचर सूचियाँ सेव हो गई हैं।

Streamlit डिप्लॉयमेंट के लिए तैयार! अब app.py चलाएँ।


In [ ]:
df

,EmpID,StartDate,Title,Supervisor,BusinessUnit,EmployeeType,PayZone,EmployeeClassificationType,TerminationType,DepartmentType,...,Current Employee Rating,Safety Measures,Tenure_Years,Age,Attrition_Target,Overall_Satisfaction_Encoded,Hygiene_Encoded,Food_Quality_Encoded,Accommodation_Encoded,Performance_Score_Encoded
0,3427,2019-09-20,Production Technician I,Peter Oneill,CCDR,Contract,Zone C,Temporary,Unk,Production,...,4,Average,5.3,55.3,0,3,1,2.0,1.0,3
1,3428,2023-02-11,Production Technician I,Renee Mccormick,EW,Contract,Zone A,Part-Time,Unk,Production,...,3,Average,1.9,NaN,0,4,3,3.0,1.0,3
2,3429,2018-12-10,Area Sales Manager,Crystal Walker,PL,Full-Time,Zone B,Part-Time,Unk,Sales,...,4,Poor,6.1,33.3,0,3,1,2.0,1.0,3
3,3430,2021-06-21,Area Sales Manager,Rebekah Wright,CCDR,Contract,Zone A,Full-Time,Unk,Sales,...,2,Poor,3.5,26.8,0,1,1,2.0,3.0,3
4,3431,2019-06-29,Area Sales Manager,Jason Kim,TNS,Contract,Zone A,Temporary,Unk,Sales,...,3,Poor,5.5,NaN,0,3,3,3.0,2.0,3
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2995,3422,2022-06-22,Production Technician I,Bethany Carter,PYZ,Part-Time,Zone C,Part-Time,Retirement,Production,...,3,Good,2.5,NaN,1,2,1,1.0,3.0,3
2996,3423,2020-12-28,Production Technician I,Caroline Harris,SVG,Part-Time,Zone A,Full-Time,Unk,Production,...,3,Average,4.0,23.6,0,4,3,2.0,3.0,3
2997,3424,2020-12-09,Production Technician I,Mr. James Castillo,TNS,Contract,Zone B,Temporary,Involuntary,Production,...,2,Poor,4.1,NaN,1,3,3,2.0,1.0,3
2998,3425,2019-05-28,Production Technician I,Michael Woods,WBL,Contract,Zone B,Full-Time,Resignation,Production,...,2,Average,5.6,NaN,1,4,2,3.0,3.0,3
